# Phase 2 — Feature Engineering
Build and inspect all feature channel groups (G0–G4) for the multi-view node tensor.
Run cells in order. Weather (G3) requires a NOAA token — set it in Cell 3 before running.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config
from src.utils.io import read_panel

crops  = pd.read_csv(config.PANEL_DIR / "crop_list.csv")["crop"].tolist()
panel  = read_panel()[crops]
print(f"Panel: {panel.shape}  |  crops: {len(crops)}")
panel.head()

## G0 — Target + Causal Lags

In [ ]:
from src.features.lag_features import make_lag_features, LAGS

lag_df = make_lag_features(panel)
print(f"Lag feature shape: {lag_df.shape}")
print(f"Lags built: {LAGS}")
print(f"NaN rows from max lag (52): {lag_df.isna().any(axis=1).sum()} — expected ~52")

# Spot-check: strawberry lag52 should correlate strongly with strawberry
r = panel["Strawberry"].corr(lag_df["Strawberry_lag52"])
print(f"
Strawberry ~ Strawberry_lag52  r = {r:.3f}  (expect > 0.7)")

## G1 + G2 — Fourier Terms and Season Flags

In [ ]:
from src.features.calendar_features import make_fourier_features, make_season_flags

fourier = make_fourier_features(panel.index)
flags   = make_season_flags(panel.index, crops)
print(f"Fourier shape: {fourier.shape}")
print(f"Season flags shape: {flags.shape}")

# Plot: sin_k1 should peak in summer, trough in winter
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
fourier[["sin_k1", "cos_k1"]].plot(ax=axes[0], title="Fourier k=1")

# Season flag for a few representative crops
sample_crops = ["Strawberry", "Watermelon", "Blueberry", "Pumpkin"]
flags[[f"{c}_in_season" for c in sample_crops]].plot(
    ax=axes[1], title="In-season flags (1 = peak FL season)", alpha=0.7)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "features_fourier_season.png", dpi=150)
plt.show()

## G3 — Weather (NOAA)
Set your NOAA token below before running. Skip if you do not have one yet —
the cache file will be created on first run and reused automatically.

In [ ]:
import os
os.environ["NOAA_TOKEN"] = "YOUR_TOKEN_HERE"   # replace before running

from src.features.weather_features import build_weather_panel, align_to_panel

weather  = build_weather_panel()          # downloads once, then cached
wx_panel = align_to_panel(weather, panel.index)

print(f"Weather shape: {wx_panel.shape}")
print(f"Missing weeks: {wx_panel.isna().any(axis=1).sum()}")

wx_panel.plot(subplots=True, figsize=(14, 8),
              title=["TMAX (°F)", "TMIN (°F)", "PRCP (mm)", "GDD"])
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "features_weather_fl.png", dpi=150)
plt.show()

## G4 — Static Node Attributes

In [ ]:
from src.features.static_attributes import get_raw_attributes, get_attribute_matrix

raw  = get_raw_attributes(crops)
attr = get_attribute_matrix(crops)

print("Raw attributes (verify against UF/IFAS before use in paper):")
print(raw.to_string())
print(f"
One-hot attribute matrix shape: {attr.shape}")
print(f"Columns: {list(attr.columns)}")

## Assemble Full Feature Tensor

In [ ]:
from src.features.build_tensor import (
    build_feature_groups, concat_groups,
    fit_crop_scalers, fit_weather_scaler
)

# --- demo: build WITHOUT scalers (EDA only) ---
# In the modelling loop, fit scalers on train slice only.
fg = build_feature_groups(
    panel   = panel,
    weather = wx_panel,
    crops   = crops,
    crop_scalers   = None,
    weather_scaler = None,
)

for name, arr in fg.items():
    print(f"  {name}: {arr.shape}  (nodes x time x channels)")

X_full = concat_groups(fg)
print(f"
Full tensor X: {X_full.shape}  (N={len(crops)} nodes, "
      f"T={panel.shape[0]} weeks, C={X_full.shape[2]} channels total)")

## STL Remainders (for Phase 4 graph construction)

In [ ]:
from src.features.stl_decomp import (
    stl_components, compute_stl_remainders, z_normalize_remainders
)

# Full-panel demo (use train fold only in Phase 4)
R  = compute_stl_remainders(panel)
Rz = z_normalize_remainders(R)

nan_crops = R.columns[R.isna().all()].tolist()
print(f"Crops with failed STL: {nan_crops if nan_crops else 'none'}")
print(f"Z-normed std range: [{Rz.std().min():.3f}, {Rz.std().max():.3f}]  (should ≈ 1.0)")

# Plot STL components for Strawberry (paper figure)
comp = stl_components(panel["Strawberry"])
fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
for ax, col in zip(axes, comp.columns):
    comp[col].plot(ax=ax, title=col)
plt.suptitle("STL decomposition — Strawberry", fontsize=13)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "stl_strawberry.png", dpi=150)
plt.show()

## Ablation preview — what changes when we drop a group?
This is a quick sanity check; the real ablation runs in Phase 6.

In [ ]:
from src.features.build_tensor import concat_groups

ablations = {
    "All groups":        ["G0","G1","G2","G3","G4"],
    "No weather (G3)":   ["G0","G1","G2","G4"],
    "No static (G4)":    ["G0","G1","G2","G3"],
    "Lags only (G0)":    ["G0"],
}
for name, inc in ablations.items():
    X = concat_groups(fg, include=inc)
    print(f"{name:<25}  X shape: {X.shape}")